# Modelos de Regresión: predicción de la popularidad musical en Spotify

**Objetivo:** estimar la popularidad (`popularity`, escala de 0 a 100) a partir de características acústicas y variables categóricas como el género.

Este notebook compara cuatro configuraciones: un árbol de decisión base, un árbol optimizado mediante `RandomizedSearchCV`, un `HistGradientBoostingRegressor` base y su versión optimizada. La evaluación se realiza sobre la misma partición de prueba para comparar sus resultados de forma consistente.

**Guía de lectura:** cada sección presenta el propósito del modelo, sus métricas y una interpretación de los resultados. La última sección reúne una comparación gráfica, diagnósticos de residuos y recomendaciones para continuar el análisis.

**Nota metodológica:** `popularity` es una medida observada de popularidad, no una medida objetiva de calidad musical. Los resultados describen asociaciones predictivas y no relaciones causales.

## **1. Configuración del Entorno (Code)**

In [ ]:
"""
Sección de Importación de Librerías y Módulos.

Las librerías se organizan según su función dentro del proyecto:
- Utilidades generales y comunicación con recursos externos.
- Manipulación y procesamiento de datos.
- Visualización para el análisis exploratorio.
- Preprocesamiento y preparación de datos para Machine Learning.

Esta organización facilita la lectura, mantenimiento y reutilización del código
durante las distintas etapas del proyecto, desde la carga y exploración de los
datos hasta su preparación para el entrenamiento de los modelos.
"""

# 1. Utilidades generales y comunicación con recursos externos
import math       # Proporciona funciones matemáticas y operaciones numéricas básicas.
import hashlib    # Permite generar funciones hash para identificar o verificar datos.
import io         # Proporciona herramientas para trabajar con datos en memoria como archivos.
import requests   # Permite realizar solicitudes HTTP y acceder a recursos externos.
import json       # Permite serializar y guardar diccionarios de hiperparámetros o configuraciones en formato JSON.


# 2. Manipulación y procesamiento de datos
import numpy as np   # Permite realizar operaciones numéricas eficientes sobre arrays y matrices.
import pandas as pd  # Permite trabajar con datos tabulares mediante Series y DataFrames.


# 3. Visualización de datos y Análisis Exploratorio (EDA)
import matplotlib.pyplot as plt  # Permite crear gráficos y visualizaciones de datos.
import seaborn as sns            # Extiende Matplotlib con visualizaciones estadísticas de alto nivel.


# 4. Machine Learning y Preprocesamiento (Scikit-Learn)
from sklearn.model_selection import train_test_split
# Permite dividir los datos en conjuntos de entrenamiento y prueba.

from sklearn.base import BaseEstimator, TransformerMixin
# Permite crear transformadores personalizados compatibles con Scikit-Learn.

from sklearn.compose import ColumnTransformer
# Permite aplicar diferentes transformaciones a distintos grupos de columnas.

from sklearn.pipeline import Pipeline
# Permite encadenar transformaciones y etapas de procesamiento en un flujo reproducible.

from sklearn.impute import SimpleImputer
# Transformador para rellenar valores faltantes (NaN) usando estrategias estadísticas.

from sklearn.preprocessing import OneHotEncoder, StandardScaler
# OneHotEncoder transforma variables categóricas en binarias; StandardScaler estandariza numéricas.

from sklearn.tree import DecisionTreeRegressor, plot_tree
# DecisionTreeRegressor implementa árboles de regresión; plot_tree dibuja la estructura del árbol.

from sklearn.ensemble import HistGradientBoostingRegressor
# Algoritmo de ensamble basado en Gradient Boosting optimizado con histogramas para grandes volúmenes de datos.

from sklearn.model_selection import RandomizedSearchCV
# Herramienta para la optimización eficiente de hiperparámetros mediante búsqueda aleatoria con validación cruzada.

from sklearn.metrics import mean_absolute_error, r2_score
# Funciones para evaluar el rendimiento: MAE (error promedio) y R2 (varianza explicada).

from sklearn.inspection import permutation_importance, PartialDependenceDisplay
# permutation_importance mide la relevancia real de cada variable; PartialDependenceDisplay visualiza su efecto marginal.


# 5. Gestión de archivos, directorios y serialización de modelos
import os
# Permite interactuar con el sistema operativo, gestionar rutas de archivos y crear directorios.

import joblib
# Permite guardar (.dump) y cargar (.load) artefactos binarios pesados como pipelines completos y modelos entrenados.

### *1.1 Descarga y Validación del set de datos*

In [ ]:
# obtener hash
url = "https://raw.githubusercontent.com/YorkiOprogramador/Evaluacion1_MachineLearning/main/data/Spotify_Tracks_Dataset.csv"
respuesta = requests.get(url)
hash_calculado = hashlib.sha256(respuesta.content).hexdigest()
print("Hash calculado:", hash_calculado)

In [ ]:
"""
Script para la descarga y validación criptográfica de un set de datos.

Este bloque de código obtiene un archivo CSV directamente desde un
repositorio de GitHub en su formato Raw. Antes de cargarlo en un
DataFrame de Pandas, utiliza la librería hashlib para generar un
hash SHA-256 del contenido descargado y lo compara con un hash de
referencia (previamente auditado).

El objetivo es garantizar la integridad completa del archivo,
asegurando que los datos no hayan sido alterados, corrompidos ni
manipulados externamente, previniendo así la ejecución de modelos
sesgados o vulnerados.
"""

# 1. Configuración de la URL
url = "https://raw.githubusercontent.com/YorkiOprogramador/Evaluacion1_MachineLearning/main/data/Spotify_Tracks_Dataset.csv"

# 2. Define tu Hash esperado (Copia aquí el hash calculado la primera vez que lo ejecutes)
hash_esperado = "b202fa49909b2d5cef71a04b1d21243cfeb36414535f2ca9272aa646721177bd"

# 3. Obtener el contenido del archivo desde la web
respuesta = requests.get(url)
contenido_crudo = respuesta.content # Obtenemos los bytes para el hash

# 4. Generar el Hash Criptográfico (SHA-256) del archivo
hash_calculado = hashlib.sha256(contenido_crudo).hexdigest()

# 5. Validación de Seguridad
if hash_calculado == hash_esperado:
    print("Integridad verificada: El archivo es auténtico y no ha sido modificado.")
    # Si es seguro, lo leemos directamente en Pandas
    df = pd.read_csv(io.StringIO(respuesta.text))
    display(df.head())
else:
    print("ALERTA DE SEGURIDAD: El contenido del archivo ha cambiado.")
    print(f"Hash que esperábamos: {hash_esperado}")
    print(f"Hash real obtenido: {hash_calculado}")

## *2. Preparación de datos para Regresión*

### *2.1 Análisis de Duplicados*

In [ ]:
"""
Script para detectar y analizar registros duplicados basados en 'track_id'.
"""

# 1. Contar el total de duplicados considerando la combinación de ambas columnas
duplicados_combinados = df.duplicated(['track_id']).sum()
print(f"Total de registros duplicados por track_id: {duplicados_combinados}")

# 2. Visualizar las filas duplicadas para entender la anomalía
# Al usar keep=False, mostramos todas las ocurrencias repetidas juntas para compararlas
df_duplicados = df[df.duplicated(subset=['track_id'], keep=False)]

print("\nDetalle de los primeros registros duplicados (ordenados para comparación):")
display(df_duplicados.sort_values(by=['track_id']).head(10))

### *2.2 Preparar la base exclusiva para Regresión*

In [ ]:
"""
Preparación de datos y preprocesamiento defensivo para modelado supervisado.

Objetivo:
Construir y transformar la matriz de características (X) y el vector objetivo (y)
para la predicción de la popularidad de canciones ('popularity'), garantizando
un aislamiento estricto entre los conjuntos de entrenamiento y prueba para
evitar cualquier forma de fuga de datos (data leakage).

Decisiones metodológicas:

1. Deduplicación preventiva (Aislamiento de observaciones):
   Se eliminan registros duplicados antes de la partición para evitar que
   observaciones idénticas queden distribuidas simultáneamente en 'train' y 'test',
   lo cual inflaría de manera artificial las métricas de evaluación del modelo.

2. Selección y clasificación de variables:
   - Target: 'popularity' (variable continua a predecir).
   - Variables Categóricas: Atributos estructurales y contextuales ('explicit',
     'key', 'mode', 'time_signature', 'track_genre').
   - Variables Numéricas: Métricas acústicas y temporales cuantitativas
     ('duration_ms', 'danceability', 'loudness', 'speechiness', 'acousticness',
     'instrumentalness', 'liveness', 'valence', 'tempo').

3. Partición Train / Test previa al ajuste:
   La división de datos (80% train / 20% test) se realiza de forma estricta
   antes de cualquier cálculo de parámetros de transformación (imputación
   o codificación) para mantener el conjunto de prueba totalmente no visto.

4. Preprocesamiento defensivo (Optimizado para algoritmos basados en árboles):
   - Sin Escalado Numérico: Se omite la estandarización (ej. StandardScaler)
     debido a que los modelos basados en árboles de decisión (Random Forest,
     XGBoost, LightGBM) son invariantes a transformaciones monotónicas de escala.
   - Imputación Defensiva: Se incorporan imputadores ('SimpleImputer') por mediana
     para numéricas y por moda para categóricas. Esto previene fallos por valores
     faltantes imprevistos durante la inferencia o en producción.
   - Codificación Categórica Robusta: Se aplica 'OneHotEncoder' configurado con
     'handle_unknown='ignore'' para absorber sin errores categorías no vistas en
     el conjunto de entrenamiento.

5. Ajuste y Transformación Aislados:
   El preprocesador aprende sus parámetros (mediana, moda, categorías) exclusivamente
   sobre 'X_train' mediante '.fit_transform()', y luego aplica únicamente la
   transformación sobre 'X_test' mediante '.transform()'.
"""


# =============================================================================
# 1. DEDUPLICACIÓN PREVENTIVA (EVITA DATA LEAKAGE)
# =============================================================================
# Se eliminan duplicados técnicos por ID
df_clean = df.drop_duplicates(subset=['track_id'])

# =============================================================================
# 2. DEFINICIÓN DE VARIABLES
# =============================================================================
target = 'popularity'

categorical_features = ['explicit', 'key', 'mode', 'time_signature', 'track_genre']
numerical_features = [
    'duration_ms', 'danceability', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

X = df_clean[categorical_features + numerical_features]
y = df_clean[target]

# =============================================================================
# 3. PARTICIÓN TRAIN / TEST
# =============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# =============================================================================
# 4. PREPROCESADOR DEFENSIVO (OPTIMIZADO PARA ÁRBOLES)
# =============================================================================
# Pipeline numérico: Imputación preventiva contra nulos (sin reescalado)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Pipeline categórico: Imputación preventiva + OneHotEncoder
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Integración de transformaciones por columna
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_features),
        ('cat', cat_pipeline, categorical_features),
    ],
    remainder='drop'
)

# =============================================================================
# 5. AJUSTE Y TRANSFORMACIÓN DE DATOS
# =============================================================================
# Aprende parámetros de imputación/codificación únicamente sobre X_train
X_train_processed = preprocessor.fit_transform(X_train)

# Aplica las transformaciones aprendidas a X_test
X_test_processed = preprocessor.transform(X_test)

# Mapeo de nombres de columnas resultantes tras One-Hot Encoding
feature_names = preprocessor.get_feature_names_out()

## *3. DecisionTreeRegressor*

### *3.1 Modelo de Regresión Base (DecisionTreeRegressor)*

In [ ]:
"""
Pipeline de Regresión con Árbol de Decisión (DecisionTreeRegressor)
===================================================================
Flujo de trabajo para el entrenamiento y evaluación del modelo de regresión:

1. Preparación de datos:
   - Copia los conjuntos de datos originales de entrenamiento y prueba (X_train,
     X_test, y_train, y_test) hacia variables dedicadas al experimento de regresión,
     conservando los datos sin winsorización.

2. Construcción del Pipeline End-to-End:
   - Preprocesamiento (`ColumnTransformer`):
       * num: Mantiene las variables numéricas sin alteraciones ('passthrough').
       * cat: Aplica las transformaciones definidas en (categorical_transformer)
         a las columnas categóricas.
       * remainder:('drop') Descarta cualquier columna no especificada.
   - Estimador (`DecisionTreeRegressor`):
       * Inicializa el árbol de decisión para regresión fijando la semilla `random_state=42`
         para asegurar reproducibilidad.
       * Limitación de la profundidad máxima del árbol (max_depth = 6) para restringir
         la complejidad del modelo y prevenir el sobreajuste sobre los datos de entrenamiento.

3. Entrenamiento y Predicción:
   - fit: Ejecuta la transformación de los datos y ajusta el árbol sobre el
     conjunto de entrenamiento (X_train_r, y_train_r).
   - predict: Transforma el conjunto de prueba (X_test_r) bajo el mismo esquema
     y genera las estimaciones numéricas correspondientes (y_pred_r).
"""

# 1. Asignar los datos del experimento seleccionado a las variables específicas de regresión
# Usamos los conjuntos de datos del experimento sin winsorizar por decisión grupal.
X_train_r = X_train.copy()
X_test_r = X_test.copy()
y_train_r = y_train.copy()
y_test_r = y_test.copy()

# 2. Configurar el Pipeline End-to-End para Regresión
pipeline_reg_final = Pipeline(steps=[
    ('preprocesador', ColumnTransformer(
        transformers=[
            ('num', num_pipeline, numerical_features),
            ('cat', cat_pipeline, categorical_features)
        ],
        remainder='drop'
    )),
    ('modelo', DecisionTreeRegressor(max_depth=6, random_state=42))
])

# 3. Entrenamiento del modelo
pipeline_reg_final.fit(X_train_r, y_train_r)

# 4. Realizar predicciones sobre el conjunto de prueba
y_pred_r = pipeline_reg_final.predict(X_test_r)

In [ ]:
"""
Evaluación del rendimiento del modelo de regresión:

1. Cálculo de Métricas:
   - Error Absoluto Medio (MAE): Compara los valores reales (y_test_r) frente a
     las predicciones (y_pred_r), cuantificando la magnitud promedio del error
     en las unidades originales de la variable objetivo.
   - Coeficiente de Determinación (R²): Mide la proporción de la varianza total de la
     variable objetivo explicada por el árbol de regresión (1.0 representa un ajuste perfecto).
"""
mae_r = mean_absolute_error(y_test_r, y_pred_r)
r2_r = r2_score(y_test_r, y_pred_r)

print("=== RENDIMIENTO DEL ÁRBOL DE REGRESIÓN BASE ===")
print(f"Error Absoluto Medio (MAE): {mae_r:.2f} pts")
print(f"Coeficiente de Determinación (R²): {r2_r:.2f}")

In [ ]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_reg_final.score(X_train_r, y_train_r):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_reg_final.score(X_test_r, y_test_r):.5f}")

### Interpretación del árbol de decisión base

El árbol base obtuvo un **MAE de 15,81 puntos** y un **R² de 0,10** en el conjunto de prueba. En promedio, sus predicciones se alejan casi 16 puntos de la popularidad observada, en una escala de 0 a 100. El R² indica que el modelo explica aproximadamente el 10 % de la variabilidad de la popularidad en datos no utilizados durante el entrenamiento.

El resultado es mejor que el de un predictor que asignara siempre la media del conjunto de entrenamiento, porque el R² es positivo; sin embargo, la capacidad explicativa sigue siendo limitada. Además, las puntuaciones de entrenamiento y prueba son muy cercanas (0,104 y 0,101). Esto **no sugiere un sobreajuste importante**, pero tampoco es una señal de buen desempeño: ambos resultados son bajos, por lo que el árbol parece demasiado simple para capturar buena parte de los patrones existentes.

La profundidad máxima de 6 limita la complejidad y facilita la interpretación, pero también puede impedir que el árbol represente interacciones más específicas entre género y características acústicas. Por eso tiene sentido probar una búsqueda de hiperparámetros en lugar de asumir que la configuración inicial es suficiente.

**Conclusión parcial:** el árbol base ofrece una referencia interpretable, pero su error es elevado para estimar con precisión la popularidad individual de una canción. La mejora debe evaluarse con el mismo conjunto de prueba y las mismas métricas.

In [ ]:
# Accede al estimador final del pipeline entrenado extrayendo el paso con la etiqueta "modelo",
# lo que permite interactuar directamente con el objeto del árbol
#(por ejemplo, para graficarlo con plot_tree o consultar sus hiperparámetros).
modelo_dtr = pipeline_reg_final.named_steps["modelo"]

In [ ]:
"""
Visualización Estructural del Árbol de Regresión Base
Flujo para la extracción de metadatos, reconstrucción de variables y graficación:

1. Configuración del Lienzo:
   - Inicializa una figura de gran formato (25x10 pulgadas) para asegurar legibilidad.

2. Inspección de Complejidad Estructural:
   - Extrae la profundidad total (`get_depth()`) y la cantidad de hojas terminales
     (`get_n_leaves()`), útiles para auditar el sobreajuste del modelo base.

3. Reconstrucción de Nombres de Variables:
   - Extrae los nombres transformados de las variables categóricas desde el codificador
     del preprocesador (named_transformers_['cat']) y los concatena con las numéricas
     para rotular las reglas de decisión con sus nombres reales.

4. Renderizado Gráfico del Árbol (plot_tree):
   - Limita la vista a los primeros niveles (`max_depth=3`) para prevenir saturación visual.
   - Aplica coloreado por magnitud de predicción (`filled=True`) y formatea los
     umbrales numéricos a dos decimales (`precision=2`).
"""

# Configurar el lienzo de visualización
fig, ax = plt.subplots(figsize=(25, 10))

# El modelo ya es un árbol individual, no requiere acceder a .estimators_
print(f"Profundidad del árbol: {modelo_dtr.get_depth()}")
print(f"Número de nodos terminales: {modelo_dtr.get_n_leaves()}")

# Recuperar los nombres de las columnas transformadas usando el preprocesador del pipeline de regresión
cat_encoder = pipeline_reg_final.named_steps['preprocesador'].named_transformers_['cat']
transformed_feature_names = numerical_features + cat_encoder.get_feature_names_out(categorical_features).tolist()

# Graficar los niveles superiores del árbol para evitar colapsar la memoria (max_depth=3)
plot = plot_tree(
    decision_tree = modelo_dtr,
    max_depth     = 3,
    feature_names = transformed_feature_names,
    filled        = True,
    impurity      = False,
    fontsize      = 10,
    precision     = 2,
    ax            = ax
)

plt.title("Estructura del Árbol de Regresión Base (Primeros Niveles)", fontsize=16)
plt.show()

### *3.2 Modelo de Regresión Optimizado (DecisionTreeRegressor)*

In [ ]:
"""
Optimización de Hiperparámetros mediante Búsqueda Aleatoria (RandomizedSearchCV)
================================================================================
Flujo de tuning optimizado para volúmenes de datos masivos (~80k registros):

1. Espacio de Distribuciones/Opciones (param_distributions):
   - Se define un rango más amplio de hiperparámetros para el paso 'modelo'.
   - modelo__max_depth: Evalúa desde profundidades bajas hasta profundas.
   - modelo__min_samples_split: Regula el tamaño mínimo para hacer divisiones.
   - modelo__min_samples_leaf: Garantiza densidad suficiente en los nodos hoja.
   - modelo__max_features: Explora el submuestreo de características.

2. Configuración del Muestreo (RandomizedSearchCV):
   - n_iter=30: Limita la búsqueda a 30 combinaciones aleatorias únicas.
   - cv=5: Validación cruzada de 5 pliegues para asegurar generalización.
   - scoring='neg_mean_absolute_error': Criterio de optimización enfocado en MAE.
   - random_state=42: Asegura que la selección aleatoria de parámetros sea reproducible.
   - n_jobs=-1: Paraleliza el cómputo en todos los procesadores disponibles.
"""

# 1. Definir la distribución de hiperparámetros (espacio de búsqueda expandido)
param_distributions = {
    'modelo__max_depth': [3, 5, 8, 10, 12, 15, 20, None],
    'modelo__min_samples_split': [2, 5, 10, 20, 50, 100],
    'modelo__min_samples_leaf': [1, 2, 5, 10, 20, 50],
    'modelo__max_features': [None, 'sqrt', 'log2', 0.5, 0.8]
}

# 2. Configurar RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=pipeline_reg_final,
    param_distributions=param_distributions,
    n_iter=30,             # Número de combinaciones a probar
    cv=5,                  # 5 pliegues -> Total: 150 ajustes
    scoring='neg_mean_absolute_error',
    random_state=42,       # Semilla para reproducibilidad en la selección aleatoria
    n_jobs=-1,
    verbose=1
)

# 3. Ejecutar la búsqueda sobre el conjunto de entrenamiento
random_search.fit(X_train_r, y_train_r)

# 4. Extraer el mejor estimador (Pipeline optimizado)
mejor_pipeline = random_search.best_estimator_

print("\n=== MEJORES HIPERPARÁMETROS ENCONTRADOS ===")
for param, val in random_search.best_params_.items():
    print(f"{param.replace('modelo__', '')}: {val}")

In [ ]:
# 5. Evaluación del modelo optimizado en el conjunto de prueba
y_pred_tuned = mejor_pipeline.predict(X_test_r)

"""
Evaluación del rendimiento del modelo de regresión:

1. Cálculo de Métricas:
   - Error Absoluto Medio (MAE): Compara los valores reales (y_test_r) frente a
     las predicciones (y_pred_r), cuantificando la magnitud promedio del error
     en las unidades originales de la variable objetivo.
   - Coeficiente de Determinación (R²): Mide la proporción de la varianza total de la
     variable objetivo explicada por el árbol de regresión (1.0 representa un ajuste perfecto).
"""
mae_r = mean_absolute_error(y_test_r, y_pred_tuned)
r2_r = r2_score(y_test_r, y_pred_tuned)

print("=== RENDIMIENTO DEL ÁRBOL DE REGRESIÓN BASE ===")
print(f"Error Absoluto Medio (MAE): {mae_r:.2f} pts")
print(f"Coeficiente de Determinación (R²): {r2_r:.2f}")

In [ ]:
print(f"{'Score del modelo en entrenamiento':<40}:{mejor_pipeline.score(X_train_r, y_train_r):.5f}")
print(f"{'Score del modelo en test': <40}:{mejor_pipeline.score(X_test_r, y_test_r):.5f}")

### Interpretación del árbol de decisión optimizado

La búsqueda aleatoria seleccionó `min_samples_split=50`, `min_samples_leaf=10`, `max_features=0.5` y `max_depth=None`. Aunque no se impuso una profundidad máxima, el tamaño mínimo de las hojas y de las divisiones restringe las decisiones basadas en muy pocas observaciones; `max_features=0.5` además limita las variables consideradas en cada división.

En el conjunto de prueba, el MAE bajó de **15,81 a 11,82 puntos** y el R² subió de **0,10 a 0,31**. La reducción del MAE es de aproximadamente **25,2 %** respecto del árbol base. El modelo optimizado comete, en promedio, un error menor y explica más variabilidad.

La puntuación de entrenamiento (0,525) supera la de prueba (0,306) en unos 0,219 puntos. Esta brecha sugiere que el modelo se adapta mejor a los datos vistos que a los nuevos; por tanto, existe cierto sobreajuste, aunque el rendimiento de prueba mejora claramente respecto de la configuración inicial.

**Conclusión parcial:** la optimización aporta una mejora sustancial, pero un R² de 0,31 indica que la mayor parte de la variabilidad de popularidad sigue sin ser explicada. Conviene contrastarlo con algún modelo de ensamble, que combina múltiples árboles de manera secuencial.

In [ ]:
# Accede al estimador final del pipeline entrenado extrayendo el paso con la etiqueta "modelo",
# lo que permite interactuar directamente con el objeto del árbol
#(por ejemplo, para graficarlo con plot_tree o consultar sus hiperparámetros).
modelo_dtr_op = mejor_pipeline.named_steps["modelo"]

In [ ]:
"""
Visualización Estructural del Árbol de Regresión Optimizado
Flujo para la extracción de metadatos, reconstrucción de variables y graficación:

1. Configuración del Lienzo:
   - Inicializa una figura de gran formato (25x10 pulgadas) para asegurar legibilidad.

2. Inspección de Complejidad Estructural:
   - Extrae la profundidad total (`get_depth()`) y la cantidad de hojas terminales
     (`get_n_leaves()`), útiles para auditar la complejidad tras la optimización.

3. Reconstrucción de Nombres de Variables:
   - Extrae los nombres transformados de las variables categóricas desde el codificador
     del preprocesador (named_transformers_['cat']) y los concatena con las numéricas
     para rotular las reglas de decisión con sus nombres reales.

4. Renderizado Gráfico del Árbol (plot_tree):
   - Limita la vista a los primeros niveles (`max_depth=3`) para prevenir saturación visual.
   - Aplica coloreado por magnitud de predicción (`filled=True`) y formatea los
     umbrales numéricos a dos decimales (`precision=2`).
"""

# Configurar el lienzo de visualización
fig, ax = plt.subplots(figsize=(25, 10))

# Mostrar estadísticas de complejidad del árbol de decisión optimizado
print(f"Profundidad del árbol optimizado: {modelo_dtr_op.get_depth()}")
print(f"Número de nodos terminales: {modelo_dtr_op.get_n_leaves()}")

# Recuperar los nombres de las columnas transformadas usando el preprocesador del pipeline optimizado
cat_encoder = mejor_pipeline.named_steps['preprocesador'].named_transformers_['cat']
transformed_feature_names = numerical_features + cat_encoder.get_feature_names_out(categorical_features).tolist()

# Graficar los niveles superiores del árbol para evitar colapsar la memoria (max_depth=3)
plot = plot_tree(
    decision_tree = modelo_dtr_op,
    max_depth     = 3,
    feature_names = transformed_feature_names,
    filled        = True,
    impurity      = False,
    fontsize      = 10,
    precision     = 2,
    ax            = ax
)

plt.title("Estructura del Árbol de Regresión Optimizado (Primeros Niveles)", fontsize=16)
plt.show()

## *4. HistGradientBoostingRegressor*


### *4.1 Modelo Base (HistGradientBoostingRegressor)*

In [ ]:
"""
Entrenamiento y evaluación del modelo HistGradientBoostingRegressor base.

Objetivo:
Construir e instanciar un Pipeline end-to-end para la transformación de datos
y el entrenamiento del modelo de regresión basado en histogramas, garantizando
reproducibilidad y protección contra valores nulos durante la inferencia.

Estructura del Pipeline:
1. Preprocesamiento (ColumnTransformer):
   - Flujo numérico: Aplicación de imputación por mediana como salvaguarda.
   - Flujo categórico: Imputación por la moda seguido de codificación One-Hot.
2. Estimador (HistGradientBoostingRegressor):
   - Modelo de ensamble por gradiente optimizado con histogramas.
   - Configuración base: 100 iteraciones de boosting y profundidad máxima de 12 niveles.
"""

# 1. Configurar el Pipeline End-to-End para HistGradientBoosting
pipeline_hgb_final = Pipeline(steps=[
    ('preprocesador', ColumnTransformer(
        transformers=[
            ('num', num_pipeline, numerical_features),
            ('cat', cat_pipeline, categorical_features)
        ],
        remainder='drop'
    )),
    ('modelo', HistGradientBoostingRegressor(
        max_iter=100,      # Número de árboles / iteraciones de boosting (equivalente a n_estimators)
        max_depth=12,      # Profundidad máxima por árbol
        random_state=42    # Semilla de reproducibilidad
    ))
])

# 2. Entrenamiento del modelo
pipeline_hgb_final.fit(X_train_r, y_train_r)

# 3. Realizar predicciones sobre el conjunto de prueba
y_pred_hgb = pipeline_hgb_final.predict(X_test_r)

In [ ]:
"""
Evaluación del Rendimiento del Modelo Base de HistGradientBoosting:

1. Cálculo de Métricas:
   - Error Absoluto Medio (MAE): Compara los valores reales (y_test_r) frente a
     las predicciones (y_pred_r), cuantificando la magnitud promedio del error
     en las unidades originales de la variable objetivo.
   - Coeficiente de Determinación (R²): Mide la proporción de la varianza total de la
     variable objetivo explicada por el árbol de regresión (1.0 representa un ajuste perfecto).
"""
mae_r = mean_absolute_error(y_test_r, y_pred_hgb)
r2_r = r2_score(y_test_r, y_pred_hgb)

print("=== RENDIMIENTO DEL HISTGRADIENTBOOSTING BASE ===")
print(f"Error Absoluto Medio (MAE): {mae_r:.2f} pts")
print(f"Coeficiente de Determinación (R²): {r2_r:.2f}")

In [ ]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_hgb_final.score(X_train_r, y_train_r):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_hgb_final.score(X_test_r, y_test_r):.5f}")

### Interpretación del HistGradientBoostingRegressor base

El HistGradientBoostingRegressor base obtiene un **MAE de 11,78 puntos** y un **R² de 0,37** en prueba. Frente al árbol base, reduce el error absoluto medio en aproximadamente **25,5 %** y eleva la proporción de variabilidad explicada de 0,10 a 0,37. Esto es coherente con la ventaja de un ensamble de boosting: cada etapa busca corregir parte de los errores acumulados por las anteriores, permitiendo representar relaciones no lineales más complejas que un único árbol.

Las puntuaciones de entrenamiento y prueba son 0,415 y 0,371, respectivamente. La brecha es relativamente acotada (0,044), lo que sugiere una generalización más estable que la observada en el árbol optimizado. No obstante, un R² de 0,37 sigue dejando sin explicar una parte considerable de las diferencias entre canciones.

**Conclusión parcial:** el boosting base supera al árbol individual tanto en MAE como en R² y muestra una brecha train-test menor. Esto lo convierte en un candidato prometedor, aunque todavía es necesario evaluar si el ajuste de hiperparámetros aporta una mejora adicional.

### *4.2 Modelo Optimizado (HistGradientBoostingRegressor)*

In [ ]:
"""
Optimización de hiperparámetros y ajuste del modelo HistGradientBoostingRegressor.

Objetivo:
Encontrar la combinación óptima de hiperparámetros para el modelo
HistGradientBoostingRegressor integrado en un Pipeline, maximizando la capacidad
de generalización y reduciendo el error medio absoluto (MAE) sobre el conjunto
de entrenamiento mediante validación cruzada.

Decisiones metodológicas:

1. Selección del Algoritmo (HistGradientBoosting):
   Se utiliza HistGradientBoosting por su alta eficiencia computacional al agrupar
   variables continuas en bins discretos (histogramas). Esto reduce drásticamente
   el tiempo de entrenamiento sin sacrificar precisión, siendo ideal para iteraciones
   de búsqueda de hiperparámetros.

2. Estrategia de Búsqueda (RandomizedSearchCV vs. GridSearchCV):
   En lugar de una búsqueda exhaustiva (GridSearchCV), se emplea búsqueda aleatoria
   ('n_iter=12') para explorar eficientemente el espacio combinatorio. Esto permite
   evaluar múltiples dimensiones de hiperparámetros en una fracción del tiempo,
   identificando regiones óptimas de rendimiento.

3. Métrica de Evaluación ('neg_mean_absolute_error'):
   Se optimiza con el Error Absoluto Medio (MAE). A diferencia del MSE o RMSE,
   el MAE es más robusto ante valores atípicos (outliers) y ofrece una interpretación
   directa y lineal del error en la misma escala que la variable objetivo ('popularity').

4. Validación Cruzada Aislada (cv=3):
   La evaluación del desempeño durante el tuning se realiza mediante validación
   cruzada de 3 pliegues (3-fold CV) aplicada exclusivamente sobre el conjunto de
   entrenamiento ('X_train_r', 'y_train_r'). Esto garantiza que el conjunto de prueba
   ('X_test_r') permanezca completamente no visto hasta la evaluación final.

5. Arquitectura del Espacio de Hiperparámetros:
   - 'max_iter' & 'learning_rate': Controlan la capacidad del ensamble y la tasa
     de aprendizaje (trade-off clásico de boosting: a menor tasa, se requieren más árboles).
   - 'max_depth' & 'min_samples_leaf': Limitan la complejidad de los árboles individuales
     y evitan divisiones en nodos con pocas observaciones, previniendo el sobreajuste.
   - 'l2_regularization': Agrega una penalización explícita sobre las hojas para
     suavizar el modelo y mejorar su capacidad de generalización.

6. Inferencia Final:
   Una vez identificada la mejor combinación, se extrae el mejor modelo ajustado
   ('best_estimator_') y se genera la predicción final sobre 'X_test_r'.
"""

# =============================================================================
# 1. DEFINICIÓN DEL ESPACIO DE HIPERPARÁMETROS
# =============================================================================
espacio_hiperparametros_hgb = {
    'modelo__max_iter': [100, 200, 300],          # Número de árboles/iteraciones de boosting
    'modelo__learning_rate': [0.01, 0.05, 0.1],   # Tasa de aprendizaje (ritmo de corrección de errores)
    'modelo__max_depth': [6, 10, 14, None],       # Profundidad máxima de cada árbol
    'modelo__min_samples_leaf': [10, 20, 50],     # Mínimo de observaciones por hoja
    'modelo__l2_regularization': [0.0, 0.1, 1.0]  # Penalización L2 para controlar sobreajuste
}

# =============================================================================
# 2. CONFIGURACIÓN DE RANDOMIZEDSEARCHCV
# =============================================================================
tuning_hgb = RandomizedSearchCV(
    estimator=pipeline_hgb_final,
    param_distributions=espacio_hiperparametros_hgb,
    n_iter=12,
    scoring='neg_mean_absolute_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# =============================================================================
# 3. EJECUCIÓN DE LA BÚSQUEDA SOBRE EL CONJUNTO DE ENTRENAMIENTO
# =============================================================================
print("Iniciando tuning del HistGradientBoostingRegressor...")
tuning_hgb.fit(X_train_r, y_train_r)

# =============================================================================
# 4. REPORTAR MEJORES HIPERPARÁMETROS
# =============================================================================
print("\n--- Resultados HistGradientBoosting Optimizado ---")
print("Mejores parámetros encontrados:")
for param, valor in tuning_hgb.best_params_.items():
    print(f"  • {param.replace('modelo__', '')}: {valor}")

# =============================================================================
# 5. EXTRAER MEJOR ESTIMADOR Y GENERAR PREDICCIONES EN TEST
# =============================================================================
mejor_pipeline_hgb = tuning_hgb.best_estimator_
y_pred_opt_hgb = mejor_pipeline_hgb.predict(X_test_r)

Iniciando tuning del HistGradientBoostingRegressor...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

--- Resultados HistGradientBoosting Optimizado ---
Mejores parámetros encontrados:
  • min_samples_leaf: 20
  • max_iter: 300
  • max_depth: 10
  • learning_rate: 0.1
  • l2_regularization: 0.1


In [ ]:
"""
Evaluación del Rendimiento del Modelo Optimizado HistGradientBoostingRegressor:

1. Cálculo de Métricas:
   - Error Absoluto Medio (MAE): Compara los valores reales (y_test_r) frente a
     las predicciones (y_pred_r), cuantificando la magnitud promedio del error
     en las unidades originales de la variable objetivo.
   - Coeficiente de Determinación (R²): Mide la proporción de la varianza total de la
     variable objetivo explicada por el árbol de regresión (1.0 representa un ajuste perfecto).
"""
mae_r = mean_absolute_error(y_test_r, y_pred_opt_hgb)
r2_r = r2_score(y_test_r, y_pred_opt_hgb)

print("=== RENDIMIENTO Optimizado Del HistGradientBoostingRegressor ===")
print(f"Error Absoluto Medio (MAE): {mae_r:.2f} pts")
print(f"Coeficiente de Determinación (R²): {r2_r:.2f}")

=== RENDIMIENTO Optimizado Del HistGradientBoostingRegressor ===
Error Absoluto Medio (MAE): 11.15 pts
Coeficiente de Determinación (R²): 0.40


In [ ]:
print(f"{'Score del modelo en entrenamiento':<40}:{mejor_pipeline_hgb.score(X_train_r, y_train_r):.5f}")
print(f"{'Score del modelo en test': <40}:{mejor_pipeline_hgb.score(X_test_r, y_test_r):.5f}")

Score del modelo en entrenamiento       :0.49891
Score del modelo en test                :0.40295


### Interpretación del HistGradientBoostingRegressor optimizado

Con la búsqueda aleatoria se seleccionaron `min_samples_leaf=20`, `max_iter=300`, `max_depth=10`, `learning_rate=0.1` y `l2_regularization=0.1`. El aumento a 300 iteraciones permite construir un ensamble más largo; el tamaño mínimo de hoja y la regularización L2 ayudan a limitar ajustes demasiado específicos.

El modelo optimizado alcanza un **MAE de 11,15 puntos** y un **R² de 0,40**. En comparación con su versión base, el MAE disminuye aproximadamente **5,4 %** (de 11,78 a 11,15) y el R² aumenta de 0,37 a 0,40. La mejora existe, pero es moderada: el ajuste de hiperparámetros refina el resultado sin transformar por completo la capacidad predictiva.

Las puntuaciones de entrenamiento y prueba son 0,499 y 0,403. La diferencia de aproximadamente 0,096 muestra cierta adaptación adicional al conjunto de entrenamiento, aunque la puntuación de prueba continúa siendo la más alta de las cuatro configuraciones evaluadas.

**Conclusión del modelo:** dentro de los modelos probados, el HistGradientBoosting optimizado presenta el mejor equilibrio entre error y capacidad explicativa. Su MAE de 11,15 significa que la predicción se desvía, en promedio, unos 11 puntos de popularidad. El R² de 0,40 también implica que alrededor del 60 % de la variabilidad observada queda sin explicar. Por tanto, es el mejor de esta comparación, pero no debe describirse como un predictor altamente preciso.

## *5. Comparación global y diagnóstico de errores*

Para elegir entre modelos no basta con mirar una única métrica. El **MAE** permite comparar el tamaño típico del error en puntos de popularidad; el **R²** indica cuánto mejora el modelo frente a una predicción constante basada en la media del conjunto de prueba. En ambos casos, se usan las predicciones del mismo conjunto de prueba para que la comparación sea justa.

Los gráficos siguientes permiten responder cuatro preguntas: ¿qué modelo tiene menor error?, ¿cuánta variabilidad explica?, ¿las predicciones siguen la tendencia de los valores reales? y ¿existe algún sesgo sistemático, como sobreestimar canciones poco populares o subestimar las más populares?

In [ ]:
"""
Consolidación de métricas, diagnóstico de sobreajuste y evaluación visual comparativa.

Objetivo:
Comparar de manera sistemática el rendimiento de los modelos desarrollados
(modelos base vs. modelos optimizados) sobre un mismo conjunto de prueba ('test'),
evaluando la precisión del pronóstico (MAE), la capacidad explicativa (R^2) y el
grado de generalización mediante el diagnóstico de brecha entre entrenamiento y prueba.

Decisiones metodológicas:

1. Evaluación Homogénea y Estándar Unificado:
   Todos los candidatos son evaluados sobre el mismo conjunto no visto ('X_test_r', 'y_test_r').
   Esto garantiza la comparabilidad directa entre arquitecturas (Árboles de Decisión vs.
   HistGradientBoosting) y entre estados de ajuste (parámetros por defecto vs. optimizados).

2. Selección de Métricas Primarias:
   - MAE (Error Absoluto Medio): Expresa la magnitud promedio del error en las
     mismas unidades de la variable objetivo ('puntos de popularidad'), siendo
     fácilmente interpretable para el negocio e insensible a distorsiones extremas.
   - R² en Test (Coeficiente de Determinación): Cuantifica la proporción de la
     varianza de la popularidad explicada por el modelo sobre datos nuevos.

3. Diagnóstico de Sobreajuste (Generalización):
   Se incluye el 'R² entrenamiento' (calculado vía .score(X_train_r, y_train_r))
   junto con el R² de test. La discrepancia entre ambas métricas permite identificar
   modelos memorizadores (alto R² en train, bajo R² en test) y validar la eficacia
   de las técnicas de regularización aplicadas durante el tuning.

4. Inspección Visual Dual:
   Se construyen gráficos de barras horizontales ordenados jerárquicamente para
   permitir una rápida lectura ejecutiva:
   - Panel Izquierdo: Minimización del error (MAE, donde menor es mejor).
   - Panel Derecho: Maximización del poder explicativo (R^2, donde mayor es mejor).
"""

# =============================================================================
# 1. CONSTRUCCIÓN DE LA TABLA COMPARATIVA DE MÉTRICAS
# =============================================================================
# Todos los modelos se evalúan sobre el mismo conjunto de prueba
metricas_modelos = pd.DataFrame([
    {
        "Modelo": "Árbol base",
        "MAE": mean_absolute_error(y_test_r, y_pred_r),
        "R²": r2_score(y_test_r, y_pred_r),
        "R² entrenamiento": pipeline_reg_final.score(X_train_r, y_train_r)
    },
    {
        "Modelo": "Árbol optimizado",
        "MAE": mean_absolute_error(y_test_r, y_pred_tuned),
        "R²": r2_score(y_test_r, y_pred_tuned),
        "R² entrenamiento": mejor_pipeline.score(X_train_r, y_train_r)
    },
    {
        "Modelo": "HistGradientBoosting base",
        "MAE": mean_absolute_error(y_test_r, y_pred_hgb),
        "R²": r2_score(y_test_r, y_pred_hgb),
        "R² entrenamiento": pipeline_hgb_final.score(X_train_r, y_train_r)
    },
    {
        "Modelo": "HistGradientBoosting optimizado",
        "MAE": mean_absolute_error(y_test_r, y_pred_opt_hgb),
        "R²": r2_score(y_test_r, y_pred_opt_hgb),
        "R² entrenamiento": mejor_pipeline_hgb.score(X_train_r, y_train_r)
    }
]).sort_values("MAE").reset_index(drop=True)

# =============================================================================
# 2. VISUALIZACIÓN COMPARATIVA DE RENDIMIENTO
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Comparativa de MAE
orden_mae = metricas_modelos.sort_values("MAE", ascending=True)
axes[0].barh(orden_mae["Modelo"], orden_mae["MAE"])
axes[0].invert_yaxis()
axes[0].set_title("Error absoluto medio (menor es mejor)")
axes[0].set_xlabel("MAE (puntos de popularidad)")
axes[0].grid(axis="x", alpha=0.25)

# Gráfico 2: Comparativa de R² en Test
orden_r2 = metricas_modelos.sort_values("R²", ascending=True)
axes[1].barh(orden_r2["Modelo"], orden_r2["R²"])
axes[1].invert_yaxis()
axes[1].set_title("Coeficiente de determinación (mayor es mejor)")
axes[1].set_xlabel("R² en test")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

# =============================================================================
# 3. REPORTE FINAL TABULAR
# =============================================================================
display(metricas_modelos.round(2))

Cómo leer la comparación

- **MAE:** cuanto más baja sea la barra, menor es la desviación absoluta media de las predicciones.
- **R²:** cuanto más alta sea la barra, mayor es la variabilidad de popularidad explicada en el conjunto de prueba. Un valor negativo indicaría que el modelo rinde peor que predecir la media de `y_test_r`.
- La elección final debe priorizar el objetivo del proyecto (el MAE) y utilizar R² como métrica complementaria.
- La diferencia entre R² de entrenamiento y prueba ayuda a detectar posible sobreajuste.

**Interpretación de la comparación global**

**Error de predicción (MAE):**
- **HistGradientBoosting optimizado** presenta el menor MAE (aprox. 11,15 puntos), por lo que es el modelo con menor error absoluto medio entre los cuatro candidatos evaluados.
- La optimización del árbol individual genera la mejora más marcada: el MAE disminuye de aproximadamente 15,81 a 11,82 puntos (reducción cercana al 25 %).
- En HistGradientBoosting, la mejora por ajuste es más moderada: de aproximadamente 11,78 a 11,15 puntos (alrededor del 5 %).

**Capacidad explicativa (R²):**
- HistGradientBoosting optimizado obtiene el mayor R² en prueba (aprox. 0,40). Esto indica que explica alrededor del 40 % de la variabilidad de la popularidad en esta partición, no que acierte el 40 % de las predicciones.
- El árbol base obtiene un R² cercano a 0,10, mientras que el árbol optimizado alcanza aproximadamente 0,31.
- La mejora de R² es consistente con una mejor adaptación a los patrones de los datos, pero el valor restante indica que la popularidad no queda explicada completamente por las variables disponibles.

**Dictamen provisional:** se selecciona HistGradientBoosting optimizado porque combina el menor MAE y el mayor R² en esta comparación. La diferencia frente al HistGradientBoosting base es real pero moderada; por eso, la elección debe leerse como el mejor resultado observado entre los modelos probados, no como evidencia de que el modelo sea altamente preciso o generalice igual en cualquier población.

In [ ]:
"""
Análisis de residuos y diagnóstico de errores para el mejor modelo de regresión.

Objetivo:
Evaluar la calidad de las predicciones y el comportamiento de los errores del modelo
con menor MAE, mediante gráficos de dispersión de valores reales frente a predichos
y análisis del sesgo y la variación en la distribución de los errores.

Decisiones metodológicas:

1. Selección Dinámica del Mejor Candidato:
   Se identifica automáticamente el modelo con el menor Error Absoluto Medio (MAE)
   sobre el conjunto de prueba ('X_test_r', 'y_test_r'), garantizando que el diagnóstico
   se concentre en la arquitectura de mejor rendimiento global.

2. Muestreo Aleatorio para Control de Superposición de Puntos (Subsampling):
   Al graficar conjuntos de datos extensos, la superposición de puntos dificulta la
   detección de patrones o densidades. Se extrae una muestra aleatoria de hasta 4,000
   observaciones con semilla fija ('seed=42') exclusivamente para la representación gráfica,
   manteniendo el cálculo de métricas sobre el 100% de la muestra de prueba.

3. Diagnóstico Valores Reales vs. Predichos (Panel Izquierdo):
   Se grafica 'Popularidad Real' frente a 'Popularidad Predicha' comparado con la línea
   diagonal de referencia (donde la predicción es perfecta). Permite detectar si el
   modelo tiende a subestimar valores muy altos o sobreestimar valores muy bajos.

4. Análisis de Errores e Invarianza (Panel Derecho):
   Se grafica el error de predicción (Error = Valor Real - Valor Predicho)
   frente a las predicciones del modelo. Esto permite diagnosticar:
   - Heterocedasticidad: Si la magnitud del error varía a lo largo de la escala de predicción.
   - Patrones no lineales: Fallas sistemáticas en la estructura del modelo.

5. Cuantificación del Sesgo Medio:
   Se reporta el promedio de los errores. Un valor cercano a cero indica
   que los errores están centrados y el modelo no presenta una desviación o sesgo sistemático hacia la sobreestimación o subestimación.
"""

# =============================================================================
# 1. SELECCIÓN DINÁMICA DEL MEJOR MODELO POR MAE
# =============================================================================
predicciones_por_modelo = {
    "Árbol base": y_pred_r,
    "Árbol optimizado": y_pred_tuned,
    "HistGradientBoosting base": y_pred_hgb,
    "HistGradientBoosting optimizado": y_pred_opt_hgb
}

nombre_mejor = min(
    predicciones_por_modelo,
    key=lambda nombre: mean_absolute_error(y_test_r, predicciones_por_modelo[nombre])
)

y_pred_mejor = predicciones_por_modelo[nombre_mejor]
residuos = np.asarray(y_test_r) - np.asarray(y_pred_mejor)

# =============================================================================
# 2. SUBMUESTREO CONTROLADO PARA VISUALIZACIÓN
# =============================================================================
# Muestra una muestra aleatoria para evitar saturación visual; las métricas siguen usando todo el test.
rng = np.random.default_rng(42)
n_muestra = min(4000, len(y_test_r))
idx_muestra = rng.choice(len(y_test_r), size=n_muestra, replace=False)

y_real_array = np.asarray(y_test_r)
y_pred_array = np.asarray(y_pred_mejor)

# =============================================================================
# 3. CONSTRUCCIÓN DE ERRORES Y RESIDUOS
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Valores Reales vs. Predichos
axes[0].scatter(y_real_array[idx_muestra], y_pred_array[idx_muestra], alpha=0.25, s=12)
lim_inf = min(y_real_array.min(), y_pred_array.min())
lim_sup = max(y_real_array.max(), y_pred_array.max())
axes[0].plot([lim_inf, lim_sup], [lim_inf, lim_sup], linestyle="--", color="red", linewidth=1.5,
             label="Predicción perfecta")
axes[0].set_xlabel("Popularidad real")
axes[0].set_ylabel("Popularidad predicha")
axes[0].set_title(f"Real vs. predicho — {nombre_mejor}")
axes[0].legend()
axes[0].grid(alpha=0.2)

# Panel 2: Residuos vs. Predicción (Heterocedasticidad / Sesgo)
axes[1].scatter(y_pred_array[idx_muestra], residuos[idx_muestra], alpha=0.25, s=12)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Popularidad predicha")
axes[1].set_ylabel("Residuo (real − predicho)")
axes[1].set_title("Residuos frente a predicción")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# =============================================================================
# 4. REPORTAR MÉTRICAS DE DIAGNÓSTICO
# =============================================================================
print(f"Modelo graficado: {nombre_mejor}")
print(f"MAE: {mean_absolute_error(y_test_r, y_pred_mejor):.2f} | "
      f"R²: {r2_score(y_test_r, y_pred_mejor):.3f} | "
      f"Sesgo medio del residuo: {residuos.mean():.2f} puntos")

**Cómo leer los gráficos de diagnóstico**

**Gráfico de valores reales frente a predichos.** La diagonal representa el escenario ideal en que cada predicción coincide con su valor real. Cuanto más cerca estén los puntos de esa línea, más precisa es la predicción. Una nube muy dispersa indica errores importantes. Si los valores altos reales quedan sistemáticamente por debajo de la diagonal, el modelo tiende a subestimar las canciones más populares; si quedan por encima, tiende a sobreestimarlas.

**Gráfico de residuos.** El residuo se calcula como `popularidad real − popularidad predicha`. Los residuos positivos indican subestimación y los negativos, sobreestimación. Idealmente, los puntos se distribuyen alrededor de cero sin formar curvas ni patrones. Una estructura visible o una dispersión que aumenta con la predicción sugiere que el modelo comete errores distintos según el nivel de popularidad.

### Interpretación de los gráficos de diagnóstico

#### 1. Valores reales frente a predichos

- **Canciones con popularidad real igual a cero:** la columna vertical en `popularidad real = 0` muestra que el modelo produce predicciones positivas para parte de estas observaciones. Esto supone sobreestimación de canciones con valor cero en el dataset. Conviene recordar que un cero registrado no necesariamente explica por sí solo por qué la canción no tiene popularidad.
- **Rango intermedio:** la nube parece seguir la tendencia general, aunque la dispersión respecto de la diagonal indica que el modelo no reproduce con exactitud la popularidad de cada canción. No debe afirmarse que este tramo es el de mayor precisión sin comparar explícitamente el MAE por intervalos.
- **Popularidad alta:** si los puntos correspondientes a valores reales altos quedan por debajo de la diagonal, el modelo subestima esos casos. La concentración de predicciones en un rango más estrecho que los valores reales es compatible con una compresión de las predicciones hacia valores intermedios.

#### 2. Residuos frente a predicción

El residuo se define como `real − predicho`: los valores negativos representan sobreestimación y los positivos, subestimación.

- **Diagonal descendente:** los casos con popularidad real igual a cero cumplen `residuo = −predicción`, por lo que forman una diagonal descendente. Es una consecuencia matemática de esos casos, no una señal independiente de un nuevo mecanismo del modelo.
- **Residuos positivos grandes:** indican observaciones cuya popularidad real fue bastante mayor que la predicción. Son compatibles con subestimación de algunas canciones populares.
- **Dispersión desigual:** si la amplitud de los residuos cambia a lo largo de las predicciones, el error depende del nivel estimado de popularidad. Esto aconseja revisar métricas por intervalos, en lugar de confiar únicamente en el MAE global.

#### Conclusión del diagnóstico

Los gráficos sugieren que el modelo tiene dificultades con los extremos de la distribución: sobreestima parte de las canciones con popularidad cero y subestima algunas de las canciones más populares. Por tanto, el mejor resultado global no implica un desempeño uniforme en todos los casos.

## *6. Interpretabilidad del modelo: importancia de variables*

Las métricas globales indican cuánto se equivoca el modelo, pero no qué variables aportan más información ni cómo cambia la predicción al variar una característica. Para complementar el diagnóstico, se utilizó la siguiente herramienta sobre el HistGradientBoosting optimizado, seleccionado por su menor MAE en test:

1. **Importancia por permutación:** mide cuánto empeora el MAE cuando se desordena una variable del conjunto de prueba. Un aumento mayor del error indica que el modelo depende más de la información contenida en esa variable. La medida es predictiva, no causal; además, variables correlacionadas pueden repartirse o sustituirse información.

Se trabaja con una muestra reproducible del conjunto de prueba para controlar el coste computacional. La muestra se usa para estos gráficos de interpretación, no para sustituir las métricas calculadas sobre el conjunto completo.

In [ ]:
"""
Análisis de Interpretabilidad Global Mediante Importancia por Permutación
========================================================================
Flujo de trabajo para evaluar la relevancia de variables sobre el mejor modelo:

1. Selección de Modelo y Submuestreo:
   - Toma el pipeline óptimo (`mejor_pipeline_hgb`).
   - Extrae una muestra representativa y reproducible de hasta 2500 registros del
     conjunto de prueba (`X_test_r`, `y_test_r`) para optimizar el coste de cálculo.

2. Cálculo de Importancia por Permutación (`permutation_importance`):
   - Desordena aleatoriamente los valores de cada variable original (`n_repeats=3`)
     y cuantifica la degradación del rendimiento usando `scoring='neg_mean_absolute_error'`.
   - La métrica resultante refleja el incremento directo en el MAE (puntos de error)
     al romper la relación entre la variable y el objetivo.

3. Estructuración y Visualización:
   - Consolida la media y dispersión (`importances_mean`, `importances_std`) en un DataFrame.
   - Genera un gráfico de barras horizontales con barras de error para las 12 variables
     más influyentes.
   - Despliega la tabla completa ordenada y redondeada a 3 decimales.
"""

# -------------------------------------------------------------------------
# 1. Definición del modelo y preparación de la muestra para interpretabilidad
# -------------------------------------------------------------------------
# Pipeline ganador según la métrica MAE para auditar su comportamiento
modelo_explicable = mejor_pipeline_hgb

# Fija un tope de 2500 observaciones para evitar tiempos de cómputo excesivos al permutar
n_interpretacion = min(2500, len(X_test_r))

# Extrae una muestra aleatoria reproducible (seed 42) del conjunto de prueba
X_interpretacion = X_test_r.sample(n=n_interpretacion, random_state=42)

# Sincroniza la variable objetivo exactamente con los mismos índices de la muestra seleccionada
y_interpretacion = y_test_r.loc[X_interpretacion.index]

# -------------------------------------------------------------------------
# 2. Cálculo de Importancia por Permutación
# -------------------------------------------------------------------------
# Evalúa el impacto de "romper" la señal de cada columna permutando sus valores
perm = permutation_importance(
    modelo_explicable,
    X_interpretacion,
    y_interpretacion,
    scoring='neg_mean_absolute_error',  # En scikit-learn scores más altos son mejores; una caída en neg_mae equivale a un aumento de MAE
    n_repeats=3,                        # Repite la permutación 3 veces por columna para medir estabilidad
    random_state=42,                    # Asegura reproducibilidad en el desorden aleatorio
    n_jobs=-1                           # Utiliza todos los núcleos de CPU disponibles en paralelo
)

# -------------------------------------------------------------------------
# 3. Tabulación y ordenamiento de los resultados
# -------------------------------------------------------------------------
# Estructura los resultados en un DataFrame vinculando el nombre de cada feature con su impacto medio y dispersión
importancias = pd.DataFrame({
    'Variable': X_interpretacion.columns,
    'Aumento MAE al permutar': perm.importances_mean,
    'Desviación estándar': perm.importances_std
}).sort_values('Aumento MAE al permutar', ascending=False)  # Ordena de mayor a menor impacto

# Extrae las 12 variables más relevantes e invierte el orden para que la principal quede en la parte superior del gráfico horizontal
top_importancias = importancias.head(12).sort_values('Aumento MAE al permutar')

# -------------------------------------------------------------------------
# 4. Generación gráfica del ranking de variables (Gráfico de barras)
# -------------------------------------------------------------------------
# Configura el tamaño del lienzo
plt.figure(figsize=(10, 6))

# Genera barras horizontales donde el largo representa el aumento del error y xerr añade las barras de error (desviación estándar)
plt.barh(
    top_importancias['Variable'],
    top_importancias['Aumento MAE al permutar'],
    xerr=top_importancias['Desviación estándar'],
    alpha=0.85
)

# Añade una línea guía vertical en x=0 para distinguir variables útiles de aquellas neutras o con ruido negativo
plt.axvline(0, color='black', linewidth=0.8)

# Rotula los ejes y asigna el título principal
plt.xlabel('Aumento del MAE tras permutar la variable (puntos)')
plt.ylabel('Variable')
plt.title('Importancia por permutación — HistGradientBoosting optimizado')

# Configura una cuadrícula tenue en el eje horizontal para facilitar la lectura numérica
plt.grid(axis='x', alpha=0.25)

# Ajusta márgenes para evitar solapamiento de textos y renderiza la figura
plt.tight_layout()
plt.show()

### Cómo interpretar la importancia por permutación

Las variables se ordenan según el aumento medio del MAE al mezclar sus valores. Una barra positiva y grande indica que destruir la asociación entre esa variable y las observaciones empeora las predicciones. Una importancia cercana a cero sugiere que, en este modelo y esta muestra, la variable aporta poca mejora adicional o que su información puede estar recogida por otras variables.

La clasificación puede variar si existen variables relacionadas entre sí; por eso no debe interpretarse como una jerarquía causal ni como una medida universal de la importancia musical.

**Interpretación de resultados:**

* Variable dominante (track_genre): Es, con una diferencia abrumadora, el factor clave de la predicción. Alterar aleatoriamente sus valores incrementa el error medio (MAE) en casi 7 puntos.   

* Variables secundarias de peso: Atributos acústicos y de duración (acousticness, instrumentalness y duration_ms) ocupan el segundo nivel de relevancia, impactando el MAE entre 0.35 y 0.50 puntos al ser permutados.

* Variables de menor / nulo impacto: Atributos como danceability, loudness, valence y speechiness tienen un impacto moderado-bajo (0.2 a 0.3 puntos), mientras que tempo, explicit, liveness y key aportan una influencia prácticamente insignificante al modelo (0.1 puntos de MAE).   

* Estabilidad del cálculo: Las barras de error de color negro en los extremos de cada columna son extremadamente cortas, lo que confirma que la importancia de cada variable es consistente y confiable entre las distintas iteraciones.

## *7. Exportación de los modelos finales y resultados de regresión*

In [51]:
# =============================================================================
# EXPORTACIÓN FINAL DE MODELOS Y RESULTADOS DE REGRESIÓN
# =============================================================================

"""
Se almacenan los pipelines de regresión completos y las métricas resultantes
para garantizar la reproducibilidad y facilitar el despliegue a producción.
"""

# 1. Crear directorio dedicado para los artefactos de regresión
directorio_exportacion = "archivos_regresion"
os.makedirs(directorio_exportacion, exist_ok=True)


# 2. Guardar los Pipelines finales (.joblib)
# Nota: Guardar el Pipeline completo preserva tanto el preprocesador como el modelo.

# Pipeline principal: HistGradientBoosting Optimizado
joblib.dump(
    mejor_pipeline_hgb,
    os.path.join(directorio_exportacion, "pipeline_hgb_optimizado.joblib")
)

# Pipeline secundario: Árbol de Decisión Optimizado
joblib.dump(
    mejor_pipeline,
    os.path.join(directorio_exportacion, "pipeline_decision_tree_optimizado.joblib")
)


# 3. Exportar tabla comparativa de rendimiento de los modelos (.csv)
# Extraemos las métricas reales del árbol de decisión optimizado y del HistGradientBoosting optimizado
mae_dt_optimizado = mean_absolute_error(y_test_r, y_pred_tuned)
r2_dt_optimizado = r2_score(y_test_r, y_pred_tuned)

df_comparativa_modelos = pd.DataFrame({
    'Modelo': ['Decision Tree (Optimizado)', 'HistGradientBoosting (Optimizado)'],
    'MAE_Test': [mae_dt_optimizado, mean_absolute_error(y_test_r, y_pred_opt_hgb)],
    'R2_Test': [r2_dt_optimizado, r2_score(y_test_r, y_pred_opt_hgb)]
})

df_comparativa_modelos.to_csv(
    os.path.join(directorio_exportacion, "comparativa_modelos.csv"),
    index=False
)


# 4. Exportar DataFrame con Predicciones vs Valores Reales (.csv)
df_predicciones = pd.DataFrame({
    'y_real': y_test_r,
    'y_pred_hgb': y_pred_opt_hgb,
    'error_absoluto': abs(y_test_r - y_pred_opt_hgb)
})

df_predicciones.to_csv(
    os.path.join(directorio_exportacion, "predicciones_test.csv"),
    index=False
)


# 5. Exportar la importancia de características por permutación (.csv)
# En este notebook, la variable que contiene el análisis es 'importancias'
if 'importancias' in locals():
    importancias.to_csv(
        os.path.join(directorio_exportacion, "importancia_caracteristicas.csv"),
        index=False
    )


# 6. Guardar los mejores hiperparámetros encontrados en JSON para ambos modelos
# Hiperparámetros HistGradientBoosting
mejores_params_hgb_clean = {
    param.replace('modelo__', ''): val
    for param, val in tuning_hgb.best_params_.items()
}

with open(os.path.join(directorio_exportacion, "mejores_hiperparametros_hgb.json"), "w") as f:
    json.dump(mejores_params_hgb_clean, f, indent=4)

# Hiperparámetros Decision Tree
mejores_params_tree_clean = {
    param.replace('modelo__', ''): val
    for param, val in random_search.best_params_.items()
}

with open(os.path.join(directorio_exportacion, "mejores_hiperparametros_tree.json"), "w") as f:
    json.dump(mejores_params_tree_clean, f, indent=4)


# =============================================================================
# CONFIRMACIÓN DE ARCHIVOS EXPORTADOS
# =============================================================================
print("=" * 80)
print("EXPORTACIÓN FINAL COMPLETADA EN:", directorio_exportacion)
print("=" * 80)

for archivo in sorted(os.listdir(directorio_exportacion)):
    print(f" - {archivo}")

EXPORTACIÓN FINAL COMPLETADA EN: archivos_regresion
 - comparativa_modelos.csv
 - importancia_caracteristicas.csv
 - mejores_hiperparametros_hgb.json
 - mejores_hiperparametros_tree.json
 - pipeline_decision_tree_optimizado.joblib
 - pipeline_hgb_optimizado.joblib
 - predicciones_test.csv


## *8. Conclusiones generales y próximos pasos*

1. **Mejor modelo dentro de la comparación:** HistGradientBoosting optimizado obtiene el menor MAE (≈ 11,15 puntos) y el mayor R² (≈ 0,40) en el conjunto de prueba. Es el candidato preferible entre las cuatro configuraciones evaluadas, aunque no puede considerarse altamente preciso para cada canción.
2. **La optimización mejora especialmente al árbol individual:** su MAE disminuye de aproximadamente 15,81 a 11,82 puntos. En HistGradientBoosting, la mejora es más moderada (de ≈ 11,78 a ≈ 11,15), por lo que parte importante de la ventaja proviene de la arquitectura de ensamble y no solo del ajuste de parámetros.
3. **La capacidad explicativa sigue siendo parcial:** un R² de 0,40 indica que el modelo explica alrededor del 40 % de la variabilidad observada en esta partición. El resto no queda explicado por el modelo; no debe atribuirse únicamente a variables ausentes, ya que también puede reflejar ruido, variabilidad no capturada y limitaciones del modelo.
4. **Los gráficos de diagnóstico sugieren errores no uniformes:** se observa sobreestimación en parte de las canciones con popularidad registrada como cero y subestimación en algunas canciones de popularidad alta. La diagonal descendente en el gráfico de residuos para los casos con valor real cero es una consecuencia algebraica de la definición del residuo. Estos patrones aconsejan evaluar el error por segmentos.
5. **Las variables complementan las métricas y ahí radica su importancia:** la importancia por permutación permite identificar qué variables deterioran más el MAE al desordenarse. Los resultados deben interpretarse como asociaciones predictivas, no como efectos causales.
6. **El género es una variable contextual que merece evaluación:** el análisis exploratorio encontró una asociación entre `track_genre` y los niveles de popularidad. La asociación exploratoria no garantiza por sí sola que el género aporte la mayor importancia predictiva; la permutación ayuda a comprobar su utilidad dentro del modelo ajustado.
7. **Alcance de los resultados:** las conclusiones se limitan a este dataset, a la partición de prueba utilizada y a las configuraciones evaluadas. No prueban causalidad ni garantizan el mismo rendimiento en otras épocas, catálogos o poblaciones.

### Recomendaciones para mejoras futuras

- Incorporar un **baseline explícito** con `DummyRegressor(strategy="mean")` para cuantificar la mejora frente a predecir siempre la media.
- Comparar **MAE por intervalos de popularidad y por género**; el error global puede ocultar segmentos con peor rendimiento.


## *9. Metodología CRISP-DM*

El proceso de regresión se estructura siguiendo las etapas de la metodología **CRISP-DM**:

*   **Business Understanding:** Identificar y estimar de forma cuantitativa la popularidad (`popularity`) de las canciones en Spotify a partir de sus atributos acústicos y contextuales, permitiendo así apoyar la toma de decisiones en la recomendación de contenidos, diseño de campañas, etc.

*   **Data Understanding:** Análisis y detección de registros duplicados técnicos basados en el `track_id` (~24,259 duplicados encontrados) y evaluación preliminar de las variables predictoras (tanto numéricas como categóricas) y su relación con el objetivo.

*   **Data Preparation:** Deduplicación preventiva de observaciones para evitar la fuga de datos (*data leakage*), partición estricta de conjuntos de entrenamiento (80%) y prueba (20%), codificación One-Hot para variables categóricas (`track_genre`, `explicit`, etc.) e imputación defensiva mediante mediana (numéricas) y moda (categóricas) sin reescalado innecesario para modelos basados en árboles.

*   **Modeling:** Configuración, entrenamiento y ajuste de hiperparámetros mediante búsqueda aleatoria (`RandomizedSearchCV` con validación cruzada de 3 y 5 pliegues) aplicados a dos arquitecturas de modelado supervisado:
    1. *DecisionTreeRegressor* (Árbol base vs. Árbol optimizado).
    2. *HistGradientBoostingRegressor* (Modelo base vs. Modelo optimizado).

*   **Evaluation:** Comparación sistemática de rendimiento sobre el conjunto de test unificado usando el Error Absoluto Medio ($MAE$) y el Coeficiente de Determinación ($R^2$), complementado con diagnóstico estructural del árbol, análisis de residuos para validar la heterocedasticidad, cálculo de sesgo medio y evaluación de interpretabilidad mediante la importancia de características por permutación.

*   **Deployment / Business Use:** Almacenamiento organizado y persistente de los pipelines de preprocesamiento y modelos optimizados en formato `.joblib`, exportación de métricas de desempeño comparativas, recopilación de hiperparámetros en formato `.json` y archivo de predicciones de prueba para facilitar el despliegue e integración en entornos de producción.

### *9.1 Problema de negocio*

El catálogo de Spotify contiene una inmensa variedad de canciones, pero predecir qué factores determinan su impacto y aceptación es un desafío complejo. El desconocimiento del potencial predictivo de los atributos acústicos y de género impide a los creadores de contenido, curadores y desarrolladores optimizar las recomendaciones de canciones y planificar lanzamientos con bases cuantitativas sólidas.

#### Objetivo de modelado (regresión)

Estimar de manera cuantitativa la popularidad (`popularity`, en escala de 0 a 100) de las canciones a partir de sus atributos acústicos (como la acústica, la instrumentalidad o la duración) y contextuales (género musical), utilizando algoritmos de aprendizaje supervisado basados en árboles para identificar las variables que mayor influencia ejercen sobre el éxito de un track.

#### KPIs de evaluación

**KPIs técnicos**
- **Error Absoluto Medio (MAE):** Cuantificar la desviación promedio de las predicciones del modelo frente al valor real de popularidad (logrando reducir el error típico a ≈ 11.15 puntos con el modelo HistGradientBoosting optimizado).
- **Coeficiente de Determinación ($R^2$):** Medir la proporción de variabilidad de la popularidad explicada por los modelos ajustados (alcanzando una capacidad explicativa del ≈ 40%).
- **Sesgo Medio de Residuos:** Monitorear que la media de los errores sistemáticos se mantenga cercana a cero (0.01 puntos) para garantizar predicciones insesgadas globalmente.
- **Importancia por Permutación:** Clasificar y validar de manera robusta la estabilidad del impacto de las variables (confirmando a `track_genre` como el factor de predicción dominante).

**KPIs de negocio propuestos para una etapa de despliegue**
- **Precisión de Recomendación Previa:** Incremento en la tasa de clics (CTR) en canciones sugeridas antes de su lanzamiento en función de la popularidad estimada.
- **Retorno de Inversión en Marketing:** Optimización del presupuesto de difusión al focalizar esfuerzos en canciones con alta popularidad estimada.
- **Desviación de Planificación Editorial:** Reducción del margen de error humano en la selección manual de canciones para playlists editoriales exitosas.

*Nota: Los KPIs de negocio corresponden a métricas de validación futuras que se medirían tras el despliegue del pipeline y no forman parte del dataset histórico analizado.*

### *9.2 Consideraciones éticas y limitaciones*

El modelado predictivo se realiza exclusivamente sobre las características técnicas de las canciones y no involucra datos personales, demográficos o de comportamiento de usuarios, lo cual previene riesgos directos de privacidad o discriminación algorítmica.

Sin embargo, los resultados predictivos deben interpretarse con cautela considerando los siguientes sesgos y limitaciones:
- **Naturaleza de la Popularidad:** La variable `popularity` no es una métrica objetiva de la calidad artística. Está fuertemente condicionada por la inversión publicitaria, dinámicas de viralización, gestión de la plataforma y sesgos de exposición del catálogo.
- **Limitación en la Clasificación de Géneros:** La deduplicación por `track_id` conserva únicamente una categoría en `track_genre`. Esto ignora la identidad multigénero de muchas canciones y puede introducir ruido o inconsistencias en la variable más determinante del modelo.
- **Capacidad Explicativa Acotada:** El coeficiente de determinación ($R^2 \approx 0.40$) advierte que aproximadamente el 60 % de la variabilidad de la popularidad responde a factores externos no medidos en este dataset (tales como la estacionalidad, el impacto de redes sociales o la fama del artista).
- **Uso Responsable de las Estimaciones:** Las predicciones de popularidad deben utilizarse como soporte analítico y no como criterios de filtrado absoluto en la gestión editorial o en decisiones de inversión artística.